In [ ]:
import pandas as pd
from sqlalchemy import create_engine
import datetime as dt
import time as tm
import numpy as np
from pymongo import MongoClient
from math import radians, cos, sin, asin, sqrt
import pymysql
from datetime import datetime,timedelta
import json
import urllib.request
import psycopg2


import warnings
warnings.filterwarnings("ignore")

In [ ]:
import os
from config import *  # all DB / API handles come from environment variables, see config.py


db = client.get_database('telematics_db')

db2 = client2.get_database('telematics_db')



db = client.get_database('telematics_db')


db2 = client2.get_database('telematics_db')



In [ ]:
gettime=lambda x: pd.Timestamp(x).timestamp()
getdate=lambda x: pd.datetime.fromtimestamp(x).date()
getDay=lambda x: pd.datetime.fromtimestamp(x).date()
dttoday=int(tm.time()-tm.time()%86400-19800)

In [ ]:
def haversine(L1,L2):
    """
    Calculate the great circle distance between two points
    on the earth (specified in decimal degrees)
    """

    lon1=L1[0]
    lat1=L1[1]
    lon2=L2[0]
    lat2=L2[1]
    # convert decimal degrees to radians
    lon1, lat1, lon2, lat2 = map(radians, [lon1, lat1, lon2, lat2])

    # haversine formula
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
    c = 2 * asin(sqrt(a))
    r = 6371000 # Radius of earth in kilometers. Use 3956 for miles
    return c * r


In [ ]:

def loc_pings(l):
    vid=l[0]
    day_start=l[1]
    day_end=l[2]

    try:
        url=GPS_API_BASE+"/gps/vehicles?vehicleIds={}&fromTime={}&toTime={}&sortOrder=&timeCondition=".format(int(vid),int(day_start),int(day_end))
       # print(url)
        data=json.load(urllib.request.urlopen(url))
        df=pd.DataFrame(data['data'][str(int(vid))])
        #print("LOC PINGS A GAYI")
        
    except Exception as e:
        #print(" loc pings issue >> " + str(e))
        df=pd.DataFrame()
    return df


In [ ]:

def hb_pings(l):
    vid=l[0]
    day_start=l[1]
    day_end=l[2]
    
    try:        
        url=GPS_API_BASE+"/heartbeat/list?vehicleIds={}&fromTime={}&toTime={}&sortOrder=&timeCondition=".format(int(vid),int(day_start),int(day_end))
        hb_data=json.load(urllib.request.urlopen(url))
        hb=pd.DataFrame(hb_data['data'][str(int(vid))])
        #print("HB AAYA")

    except Exception as e:
        #print("hb pings issue >> " + str(e))
        hb=pd.DataFrame()        
    return hb



In [ ]:
def analysis_flow(l):
    lc=loc_pings(l)
    hb=hb_pings(l)

    lc=lc.copy()
    hb=hb.copy()
    
    if (len(hb)>5) & (len(lc)>5):
        day_wire_issue=hb_lc_wire_issue(hb,lc)
        return day_wire_issue
    elif (len(lc)>5) & (len(hb)<1):
        day_wire_issue=lc_only_wire_issue(lc)
        return day_wire_issue
        
    else:
        print("error function : analysis_flow ; not_eough_data in heartbeat and location pings" + str(l))
        return None


In [ ]:
def hb_lc_wire_issue(hbc,dfc):
    hb=hbc.copy()
    df=dfc.copy()
    #if 1>0:
    try:

        hb.sort_values('created',inplace=True)
    
        hb.created=hb.created/1000
        hb.created=hb.created.apply(int)
        del df['created']
        del df['vehicleId']
        del hb['updated']
        df.sort_values('time',inplace=True)
    
        if len(hb)<len(df):
        
            df=pd.merge_asof(left=df,right=hb,left_on='time',right_on='created',direction='nearest')
        else:
            df=pd.merge_asof(left=hb,right=df,left_on='created',right_on='time',direction='nearest')
        df.loc[:,'loc2']=df['location'].shift(1)
        df.loc[:,'time1']=df['time'].shift(-1)
        df=df[1:]
        time_diff_cons=df['time1']-df['time']
        df.loc[:,'time_diff_cons_pings']=time_diff_cons
    
    
        loc_list=df[['location','loc2']].values.tolist()
        dis=[]
        for loc in loc_list:
            y=haversine(loc[0],loc[1])
            dis.append(y)

        df.loc[:,'distance']=dis
    
    
        df.loc[:,'speed_self_cal']=df['distance']/df['time_diff_cons_pings']*3.6    
        df.loc[:,'moving_status']=df.apply(lambda row: 'running' if row['speed_self_cal']>5 else  'stop' ,axis=1)
    
        driving_time=df[df['moving_status']=='running']['time_diff_cons_pings'].sum()

        stop_time=df[df['moving_status']=='stop']['time_diff_cons_pings'].sum()
        day=getDay((df['time'].min()+df['time'].max())/2)
    
        #### WIRE CUT KE PARAMETERS ####
        charge_off_Speed=df[df['chargeOn']==False]['speed_self_cal'].describe().values[3]
        wire_cut_time=df[df['chargeOn']==False]['time_diff_cons_pings'].sum()
        min_voltage_pct_when_wire_cut=df[df['chargeOn']==False]['voltPct'].describe().values[3]
        #print("charge_off_Speed:"+str(charge_off_Speed)+" ,wire_cut_time:"+str(wire_cut_time)+" ,min_voltage_pct_when_wire_cut:"+str(min_voltage_pct_when_wire_cut))

        #### LOOSE WIRE ####
        fluctuations=np.sum(abs(df['chargeOn']-df['chargeOn'].shift(1)))
        
        #### CHARGE CONNECTED TO IGN ####
    
        charge_to_ign=(len(list(df['chargeOn'].eq(df['engineOn']).value_counts().values))==1)&(list(df['chargeOn'].eq(df['engineOn']).value_counts().keys())[0]==True)&(fluctuations>2)      
    
        #### IGN WIRE CUT ####
        ign_wire_cut=(len(df[df['engineOn']==False][df['speed_self_cal']>10])>20) & (len(df[df['engineOn']==True])<1)
    
        #### IGN WIRE TO CHARGE ####
        ign_to_charge=(len(df[df['chargeOn']==True][df['engineOn']==False])==0 )
    
        intentional_wire_cut=(charge_off_Speed>10/3.6)&(wire_cut_time>3600)&(min_voltage_pct_when_wire_cut<60)
        maintainence_wire_cut=(charge_off_Speed<2/3.6)&(wire_cut_time>3600)&(min_voltage_pct_when_wire_cut<60)
        wire_loose=fluctuations>2
        vehicle_id=df.vehicleId.iloc[0]
        values=[day,vehicle_id,intentional_wire_cut,maintainence_wire_cut,wire_loose,charge_to_ign,ign_wire_cut,ign_to_charge]
        columns=['day','vehicle_id','intentional_wire_cut','maintainence_wire_cut','wire_loose','charge_to_ign','ign_wire_cut','ign_to_charge']
    
        wire_issues=pd.DataFrame([values],columns=columns)
        #print("______________ wireissue main gaya_________________")
        return wire_issues
 #   else:
        
    except Exception as e:
        print("The issue with the wire_issue_hb_lc {} is--- ".format(vehicle_id)+ str(e))
        return None
    

In [ ]:
def lc_only_wire_issue(dfc):
    df=dfc.copy()
   
  #  if 1>0:
    try:

        df.sort_values('time',ascending=False,inplace=True)

        df.loc[:,'loc2']=df['location'].shift(1)
        df.loc[:,'time1']=df['time'].shift(1)
        df=df[1:]
        time_diff_cons=df['time1']-df['time']
        df.loc[:,'time_diff_cons_pings']=time_diff_cons
   
   
        loc_list=df[['location','loc2']].values.tolist()

        dis=[]
        for loc in loc_list:
            y=haversine(loc[0],loc[1])
            dis.append(y)
 
        df.loc[:,'distance']=dis
   
   
   
        df.loc[:,'speed_self_cal']=df['distance']/df['time_diff_cons_pings']*3.6    
        df.loc[:,'moving_status']=df.apply(lambda row: 'running' if row['speed_self_cal']>5 else  'stop' ,axis=1)
   
        driving_time=df[df['moving_status']=='running']['time_diff_cons_pings'].sum()

        stop_time=df[df['moving_status']=='stop']['time_diff_cons_pings'].sum()

        day=getDay((df['time'].min()+df['time'].max())/2)
   
   
        #### WIRE CUT KE PARAMETERS ####
        charge_off_Speed=df[df['chargeOn']==False]['speed_self_cal'].describe().values[3]
        wire_cut_time=df[df['chargeOn']==False]['time_diff_cons_pings'].sum()
        min_voltage_pct_when_wire_cut=df[df['chargeOn']==False]['voltPct'].describe().values[3]
        #print(min_voltage_pct_when_wire_cut)

   
        #### LOOSE WIRE ####
        fluctuations=np.sum(abs(df['chargeOn']-df['chargeOn'].shift(1)))

        #### CHARGE CONNECTED TO IGN ####
        charge_to_ign=(len(list(df['chargeOn'].eq(df['ignOn']).value_counts().values))==1)&(list(df['chargeOn'].eq(df['ignOn']).value_counts().keys())[0]==True)&(fluctuations>2)

   
        #### IGN WIRE CUT ####
        ign_wire_cut=(len(df[df['ignOn']==False][df['speed_self_cal']>10])>20) & (len(df[df['ignOn']==True])<1)
   
        #### IGN WIRE TO CHARGE ####
        ign_to_charge=(len(df[df['chargeOn']==True][df['ignOn']==False])==0 )&(fluctuations<3)
   
        intentional_wire_cut=(charge_off_Speed>12/3.6)&(wire_cut_time>3600)&(min_voltage_pct_when_wire_cut<60)
        maintainence_wire_cut=(charge_off_Speed<2/3.6)&(wire_cut_time>3600)&(min_voltage_pct_when_wire_cut<60)
        wire_loose=fluctuations>2
        vehicle_id=df.vehicleId.iloc[0]
        values=[day,vehicle_id,intentional_wire_cut,maintainence_wire_cut,wire_loose,charge_to_ign,ign_wire_cut,ign_to_charge]
        columns=['day','vehicle_id','intentional_wire_cut','maintainence_wire_cut','wire_loose','charge_to_ign','ign_wire_cut','ign_to_charge']
   
        wire_issues=pd.DataFrame([values],columns=columns)
       
        return wire_issues
#    else:
           
    except Exception as e :
        print("The issue with the function lc_only_wire_issue for vehicle_id {} is --- ".format(vehicle_id)+ str(e))

        return None

In [ ]:
wire_issue_custom_ping?

In [ ]:
def wire_issue_custom_ping(vid_start_end):
#    if 1>0:  
    try:
        k=0    
        import multiprocessing
        pool = multiprocessing.Pool()
        results2=pool.map(analysis_flow,vid_start_end)
        pool.close()
        pool.join()
        
        try:
            wire_data=pd.concat(results2)
        
            #db = client.get_database('telematics_db')

            #vehicle_details=pd.read_sql("select vehicle_id,boarding_time,vehicle_number,data_source,operator_id,new_device,installation_subtype,device_id,is_force_live,operator_name,sim_id,model_name,sales_dept,sim_number,sim_phone_number,imei from analytics.vehicle_details where vehicle_id in {} and vehicle_state='LIVE' ".format(tuple(wire_data['vehicle_id'])),con=con)

            #wire_data=pd.merge(wire_data,vehicle_details,on='vehicle_id',how='left')
            #devices=pd.read_sql("select created as device_created_time,imei,new_device as device_table_new_device,version as device_version,purchase_date as device_purchase_date from fleet_lake.devices",con=con)
            #wire_data=pd.merge(wire_data,devices, on='imei',how='left')

        except Exception as e:
            print("some issue in wire_issue_custom_ping after wire_data was captured but during mapping vehilce details it failed, error being : ",e)
            wire_data=pd.DataFrame()
            return wire_data

    except Exception as e:
#    else:
        print("some issue wire_issue_custom_ping while trying multithreading on analysis flow; the error being >>"+ str(e))
        wire_data=pd.DataFrame()
    return wire_data 


def find_last_ping(vid):
    try:
        url=GPS_API_BASE+"/gps/top?vehicleIds={}&sortOrder=desc".format(vid)
        data=json.load(urllib.request.urlopen(url))
        last_loc=data['data'][str(vid)]['time']
    except:
        last_loc=0
    try:
        url=GPS_API_BASE+"/heartbeat/top?vehicleIds={}&sortOrder=desc".format(vid)
        data=json.load(urllib.request.urlopen(url))
        last_hb=data['data'][str(vid)]['time']
    except:
        last_hb=0
        
        
    if last_hb>=last_loc:
        last_ping=last_hb
    else:
        last_ping=last_loc
    
    if  last_ping>0:
        l=[[vid,last_ping-86400,last_ping+100]]
    else:
        l=[[vid,0,200]]

    return l

def wire_issue_last_ping(event,context):  #NOT A COMPLETE FUNCTION YET
####    if 1>0:
    
    try:
        k=0
        vehicle_list=event
    
                
        from multiprocessing.dummy import Pool as ThreadPool
        pool = ThreadPool(10)
        reuslts=pool.map(find_last_ping,vehicle_list)
        pool.close()
        pool.join()

        vid_start_end_draft=[]
        for i in reuslts:
            for p in i:
                vid_start_end_draft.append(p)
        
        vid_start_end=[]
        for i in vid_start_end_draft:
            if (i not in vid_start_end)&(isinstance(i,list)==True):
                vid_start_end.append(i)
        
        print((dt.datetime.now()+timedelta(hours=5.5)).strftime('%m/%d %H:%M:%S') +' :total iterations to go - '+str(len(vid_start_end)))
        from multiprocessing.dummy import Pool as ThreadPool
        pool = ThreadPool(100)
        results2=pool.map(analysis_flow,vid_start_end)
        pool.close()
        pool.join()
        
        data=pd.DataFrame()
        wire_data=pd.DataFrame()
        for x in results2:
            if x!=None:
                
                wire_data=pd.concat([wire_data,x[1]])
                data=pd.concat([data,x[0]])
    

        db = client.get_database('telematics_db')
        
                                                                                                                     

                                                                                                                      
        data=data.round(0)

        vehicle_details=pd.read_sql("select vehicle_id,boarding_time,vehicle_number,data_source,operator_id,new_device,installation_subtype,device_id,is_force_live,operator_name,operator_phone_number,sim_id,model_name,sales_dept,sim_number,sim_phone_number,imei from vehicle_details",con=engine)

        data=pd.merge(data,vehicle_details,on='vehicle_id',how='left')
        wire_data=pd.merge(wire_data,vehicle_details,on='vehicle_id',how='left')

        devices=pd.read_sql("select created as device_created_time,imei,new_device as device_table_new_device,version as device_version,purchase_date as device_purchase_date from devices",con=mysql_vds)

        data=pd.merge(data,devices, on='imei',how='left')
        wire_data=pd.merge(wire_data,devices, on='imei',how='left')


        print((dt.datetime.now()+timedelta(hours=5.5)).strftime('%m/%d %H:%M:%S')+"  " +str(k)+ "range vehicles have total data for:"+str(len(data))+" vehicles to be appended")

    except Exception as e:
####    else:
        print("KOI ISSUE AAYA HAI, VEHICLES NAHI HUE HONGE, the error being >>"+ str(e))
        data=pd.DataFrame()
    return (data,wire_data)

def wire_issue_last_analysis(vehicle_list):
####    if 1>0:
    
    try:
        k=0
        vehicle_list=event
    
                
        from multiprocessing.dummy import Pool as ThreadPool
        pool = ThreadPool(10)
        reuslts=pool.map(custom_ping,vehicle_list)
        pool.close()
        pool.join()

        vid_start_end_draft=[]
        for i in reuslts:
            for p in i:
                vid_start_end_draft.append(p)
        
        vid_start_end=[]
        for i in vid_start_end_draft:
            if (i not in vid_start_end)&(isinstance(i,list)==True):
                vid_start_end.append(i)
        
        print((dt.datetime.now()+timedelta(hours=5.5)).strftime('%m/%d %H:%M:%S') +' :total iterations to go - '+str(len(vid_start_end)))
        from multiprocessing.dummy import Pool as ThreadPool
        pool = ThreadPool(100)
        results2=pool.map(analysis_flow,vid_start_end)
        pool.close()
        pool.join()
        
        data=pd.DataFrame()
        wire_data=pd.DataFrame()
        for x in results2:
            if x!=None:
                
                wire_data=pd.concat([wire_data,x[1]])
                data=pd.concat([data,x[0]])
    

        db = client.get_database('telematics_db')
        
                                                                                                                     

                                                                                                                      
        vehicle_details=pd.read_sql("select vehicle_id,boarding_time,vehicle_number,data_source,operator_id,new_device,installation_subtype,device_id,is_force_live,operator_name,operator_phone_number,sim_id,model_name,sales_dept,sim_number,sim_phone_number,imei from vehicle_details",con=engine)

        wire_data=pd.merge(wire_data,vehicle_details,on='vehicle_id',how='left')

        devices=pd.read_sql("select created as device_created_time,imei,new_device as device_table_new_device,version as device_version,purchase_date as device_purchase_date from devices",con=mysql_vds)

        wire_data=pd.merge(wire_data,devices, on='imei',how='left')


        print((dt.datetime.now()+timedelta(hours=5.5)).strftime('%m/%d %H:%M:%S')+"  " +str(k)+ "range vehicles have total data for:"+str(len(data))+" vehicles to be appended")

    except Exception as e:
####   else:
        print("KOI ISSUE AAYA HAI, VEHICLES NAHI HUE HONGE, the error being >>"+ str(e))
        data=pd.DataFrame()
    return (data,wire_data)